In [1]:
#premodel and seed generation implementation:

import numpy as np
import random

seed = 7
np.random.seed(seed) # for numpy
random.seed(seed) # for random and other libraries
print(f'Global Random seed set to: {seed}')

Global Random seed set to: 7


## Problem 1: Automatic Gift Recognizer
- uses data_proccesing
- uses model_builder
- uses model_selector
- uses eval_performance

In [2]:
from src.data_proccesing import explore_data, showimage , split_data_into_3sets
# loading and displaying data
dataset = np.load("data/dataset.npz")
X, y = dataset["X"], dataset["y"]
explore_data(X,y) # prints class distribution
X_train, y_train, X_val, y_val, X_test, y_test = split_data_into_3sets(X, y, randomseed = seed) #splits data into train, val and test
#showimage(X_train, 5) # shows first 5 images in training set

Shape of X: (13067, 400)
Shape of y: (13067,)
Number of classes: 15
Unique labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
    Class  Count
0       0    546
1       1    900
2       2    804
3       3    454
4       4   1022
5       5   1362
6       6    858
7       7    557
8       8    888
9       9    834
10     10    782
11     11   1601
12     12    702
13     13    884
14     14    873


**Our implementation with 3 k folds**

In [ ]:
from src.model_builder import buildmodel
from src.model_trainer import train_model_cv
from src.model_selector import evaluate_models
from itertools import product # https://docs.python.org/3/library/itertools.html#itertools.product

# Source for sklearn model parameters:
# RandomForestClassifier: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
# SVC: https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html
# LogisticRegression: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
# KNeighborsClassifier: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html
#defining hyperparameter grids for each model:

rf_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10],
    'max_features': ['sqrt', 'log2']
}
svm_grid = {
    'C': [0.1, 10],
    'kernel': ['linear', 'rbf'],
}
lr_grid = {
    'C': [0.1, 10],
}
knn_grid = {
    'n_neighbors': [3, 7]
}
# Example models
rf = buildmodel('RF', n_estimators=100, max_depth=10)
svm = buildmodel('SVM', kernel='rbf', C=1)
lr = buildmodel('LR', solver='liblinear')
knn = buildmodel('KNN', n_neighbors=5)
# loop through all of the model instances with different hyperparameters defined above
# use the model trainer and model selector here to find the best model and hyperparameters
model_grids = {
    'RF': rf_grid,
    'SVM': svm_grid,
    'LR': lr_grid,
    'KNN': knn_grid
}

best_individ_models = []

for model_name, param_grid in model_grids.items():
    for params in product(*param_grid.values()):
        kwargs = dict(zip(param_grid.keys(), params))
        model = buildmodel(model_name, **kwargs)
        score = train_model_cv(model, X_train, y_train, X_val, y_val, cv=5, seed=seed)
        best_individ_models.append((model_name, kwargs, score))
    
evaluate_models(best_individ_models)






Mean accuracy over 5 folds: 0.7328
Mean accuracy over 5 folds: 0.7244
Mean accuracy over 5 folds: 0.7090
Mean accuracy over 5 folds: 0.6993
Mean accuracy over 5 folds: 0.7290
Mean accuracy over 5 folds: 0.7260
Mean accuracy over 5 folds: 0.7413
Mean accuracy over 5 folds: 0.7335
Mean accuracy over 5 folds: 0.7128
Mean accuracy over 5 folds: 0.7030
Mean accuracy over 5 folds: 0.7387
Mean accuracy over 5 folds: 0.7363
Mean accuracy over 5 folds: 0.7477


KeyboardInterrupt: 

SKLearns implementation of GridSearchCV

In [3]:
from sklearn.model_selection import GridSearchCV
from src.model_builder import buildmodel

models_and_grids = {
    'RF': (buildmodel('RF'), {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'max_features': ['sqrt', 'log2']
    }),
    'SVM': (buildmodel('SVM'), {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf']
    }),
    'LR': (buildmodel('LR'), {
        'C': [0.1, 1, 10],
        'solver': ['liblinear']
    }),
    'KNN': (buildmodel('KNN'), {
        'n_neighbors': [3, 5, 7]
    })
}

best_models = {}

for name, (model, param_grid) in models_and_grids.items():
    print(f"\nTuning {name}...")
    grid = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
    grid.fit(X_train, y_train)
    print(f"Best {name} params: {grid.best_params_}")
    print(f"Best {name} score: {grid.best_score_:.4f}")
    best_models[name] = grid.best_estimator_



Tuning RF...
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best RF params: {'max_depth': 20, 'max_features': 'sqrt', 'n_estimators': 200}
Best RF score: 0.7421

Tuning SVM...
Fitting 5 folds for each of 6 candidates, totalling 30 fits


KeyboardInterrupt: 